# 🐱 さとりねこ 中割りメーカー（ToonCrafter）
2枚の絵（動きの最初と最後）の**間の絵をAIが自動で描いて**、アニメにします。

## 使い方
1. 上のメニュー **ランタイム → ランタイムのタイプを変更 → 「T4 GPU」** を選んで保存
2. **① 準備** の ▶ を押す（初回5〜10分。終わるまで待つ）
3. **② 生成** の ▶ を押す → 絵をアップロード（2枚以上。ファイル名順につながります 例: 01.png, 02.png）
4. できた動画が表示＆自動ダウンロードされます。②は何回でもやり直せます

※ 出力は 512×320。絵の比率が違うときは中央を切り抜きます。

In [ ]:
#@title ① 準備（最初に1回だけ）
import os, subprocess, sys, time, urllib.request
os.chdir('/content')
if not os.path.exists('ComfyUI'):
    !git clone -q https://github.com/comfyanonymous/ComfyUI
    !git clone -q https://github.com/kijai/ComfyUI-DynamiCrafterWrapper ComfyUI/custom_nodes/ComfyUI-DynamiCrafterWrapper
    !pip -q install -r ComfyUI/requirements.txt
    !pip -q install -r ComfyUI/custom_nodes/ComfyUI-DynamiCrafterWrapper/requirements.txt
    !pip -q install huggingface_hub
# 互換性修正（新しいComfyUI向け）
_f='/content/ComfyUI/custom_nodes/ComfyUI-DynamiCrafterWrapper/nodes.py'
_s=open(_f).read().replace('init_empty_weights() if is_accelerate_available','init_empty_weights() if False'); open(_f,'w').write(_s)
from huggingface_hub import hf_hub_download
import shutil
M='/content/ComfyUI/models'
def get(repo, fn, dst_dir, dst_name):
    os.makedirs(dst_dir, exist_ok=True); dst=os.path.join(dst_dir, dst_name)
    if os.path.exists(dst): return
    p=hf_hub_download(repo, fn); shutil.copy(p, dst)
print('モデルをダウンロード中...')
get('Kijai/DynamiCrafter_pruned','tooncrafter_512_interp-pruned-fp16.safetensors',f'{M}/checkpoints/dynamicrafter','tooncrafter_512_interp-pruned-fp16.safetensors')
get('Kijai/CLIPVisionModelWithProjection_fp16','CLIP-ViT-H-fp16.safetensors',f'{M}/clip_vision','CLIP-ViT-H-fp16.safetensors')
ok=False
for repo in ['stabilityai/stable-diffusion-2-1','sd2-community/stable-diffusion-2-1','Manojb/stable-diffusion-2-1-base']:
    try:
        get(repo,'text_encoder/model.fp16.safetensors',f'{M}/text_encoders','sd21_clip_fp16.safetensors'); ok=True; break
    except Exception as e: print('  ',repo,'→ 取得失敗、次を試します')
if not ok: raise SystemExit('テキストエンコーダーが取れませんでした。このエラーをClaudeに伝えてください')
print('ComfyUIを起動中...')
log=open('/content/comfy.log','w')
srv=subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--disable-auto-launch','--disable-dynamic-vram'],cwd='/content/ComfyUI',stdout=log,stderr=subprocess.STDOUT)
for _ in range(120):
    try: urllib.request.urlopen('http://127.0.0.1:8188/system_stats',timeout=2); break
    except Exception: time.sleep(2)
else: raise SystemExit('起動に失敗しました。/content/comfy.log の最後をClaudeに伝えてください')
print('✅ 準備完了！ ② に進んでください')

In [ ]:
#@title ② 生成（絵をアップロード）
動きの説明 = "a cute white cat chopping mushrooms with a knife, anime"  #@param {type:"string"}
中割りの枚数 = 16  #@param {type:"slider", min:8, max:32, step:8}
ステップ数 = 20  #@param {type:"slider", min:10, max:50, step:5}
シード = 123  #@param {type:"integer"}
fps = 8  #@param {type:"slider", min:6, max:24, step:1}
import json, uuid, glob, urllib.request, time, os, subprocess
from google.colab import files
from PIL import Image, ImageOps
from IPython.display import HTML, display
from base64 import b64encode
up=files.upload()
names=sorted(up.keys())
if len(names)<2: raise SystemExit('2枚以上アップロードしてください')
IN='/content/ComfyUI/input'; os.makedirs(IN,exist_ok=True)
keys=[]
for i,n in enumerate(names):
    im=Image.open(n).convert('RGB'); im=ImageOps.fit(im,(512,320),Image.LANCZOS)
    k=f'key_{i:02d}.png'; im.save(f'{IN}/{k}'); keys.append(k)
wf={
 "1":{"class_type":"CLIPLoader","inputs":{"clip_name":"sd21_clip_fp16.safetensors","type":"stable_diffusion"}},
 "2":{"class_type":"CLIPTextEncode","inputs":{"text":動きの説明,"clip":["1",0]}},
 "3":{"class_type":"CLIPTextEncode","inputs":{"text":"blurry, distorted, deformed, text, watermark","clip":["1",0]}},
 "4":{"class_type":"CLIPVisionLoader","inputs":{"clip_name":"CLIP-ViT-H-fp16.safetensors"}},
 "5":{"class_type":"DynamiCrafterModelLoader","inputs":{"ckpt_name":"dynamicrafter/tooncrafter_512_interp-pruned-fp16.safetensors","dtype":"fp16","fp8_unet":False}},
}
for i,k in enumerate(keys): wf[f"L{i}"]={"class_type":"LoadImage","inputs":{"image":k}}
prev=["L0",0]
for i in range(1,len(keys)):
    wf[f"B{i}"]={"class_type":"ImageBatch","inputs":{"image1":prev,"image2":[f"L{i}",0]}}; prev=[f"B{i}",0]
wf["9"]={"class_type":"ToonCrafterInterpolation","inputs":{"model":["5",0],"clip_vision":["4",0],"positive":["2",0],"negative":["3",0],"images":prev,
  "steps":ステップ数,"cfg":7.0,"eta":1.0,"frames":中割りの枚数,"seed":シード,"fs":10,"vae_dtype":"fp16"}}
wf["10"]={"class_type":"ToonCrafterDecode","inputs":{"model":["5",0],"latent":["9",0],"vae_dtype":"fp16"}}
pref='tc_'+uuid.uuid4().hex[:6]
wf["11"]={"class_type":"SaveImage","inputs":{"images":["10",0],"filename_prefix":pref}}
req=urllib.request.Request('http://127.0.0.1:8188/prompt',data=json.dumps({"prompt":wf}).encode(),headers={'Content-Type':'application/json'})
try: pid=json.loads(urllib.request.urlopen(req).read())['prompt_id']
except urllib.error.HTTPError as e: raise SystemExit('ワークフローエラー: '+e.read().decode()[:1500])
print('生成中...（T4だと数分かかります）'); t0=time.time()
while True:
    time.sleep(5)
    h=json.loads(urllib.request.urlopen(f'http://127.0.0.1:8188/history/{pid}').read())
    if pid in h:
        st=h[pid].get('status',{})
        if st.get('status_str')=='error': raise SystemExit('生成エラー: '+json.dumps(st.get('messages',[]))[-1500:]+'\n→ このエラーをClaudeに伝えてください')
        break
    print(f'  {int(time.time()-t0)}秒経過', end='\r')
out=f'/content/{pref}.mp4'
subprocess.run(['ffmpeg','-y','-loglevel','error','-framerate',str(fps),'-pattern_type','glob','-i',f'/content/ComfyUI/output/{pref}_*.png','-c:v','libx264','-pix_fmt','yuv420p','-vf','scale=1024:640:flags=lanczos',out],check=True)
print(f'✅ 完成！ {int(time.time()-t0)}秒')
display(HTML(f'<video width=640 controls loop autoplay><source src="data:video/mp4;base64,{b64encode(open(out,"rb").read()).decode()}"></video>'))
!cd /content/ComfyUI/output && zip -q -j /content/{pref}_frames.zip {pref}_*.png
files.download(out); files.download(f'/content/{pref}_frames.zip')

## うまくいかないとき
- **「CUDA out of memory」** → ② の「中割りの枚数」を8にして再実行
- **エラーが出た** → 赤いメッセージをコピーしてClaudeに送ってください
- **しばらく放置して止まった** → 無料枠の時間切れ。① からやり直し